# 01 — Exploratory Data Analysis

Dataset: [Store Item Demand Forecasting Challenge](https://www.kaggle.com/competitions/demand-forecasting-kernels-only/data) — daily unit sales, 10 stores × 50 items, 2013–2017.

Goals:
1. Understand overall shape/coverage of the data.
2. Look at trend + seasonality for a representative series.
3. Quantify demand variability across all 500 (store, item) series — this matters later: high-variability SKUs need more safety stock and are the ones where Prophet tuning (vs. defaults) actually moves the needle.
4. Sanity-check stationarity assumptions (informative, even though Prophet doesn't require stationarity the way ARIMA does).

This notebook sets up the *why* behind the modeling choices in `02_prophet_forecasting.ipynb`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

from src.data_loader import get_series, list_series_keys, load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

In [ ]:
df = load_raw()
print(df.shape)
print(df["date"].min(), "->", df["date"].max())
print(f"{df['store'].nunique()} stores x {df['item'].nunique()} items")
df.head()

## Overall sales volume over time

Aggregate across all stores/items first, just to see the macro trend + seasonal pattern before drilling into individual series.

In [ ]:
daily_total = df.groupby("date", as_index=False)["sales"].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_total["date"], daily_total["sales"], linewidth=0.8)
ax.set_title("Total daily sales across all stores/items")
ax.set_ylabel("units")
plt.tight_layout()

## One representative series in depth

Pick store 1 / item 1 and decompose it into trend, weekly seasonality, and residual using STL. This is the pattern Prophet needs to fit well — a clear upward trend plus strong weekly (and likely annual) seasonality is exactly Prophet's sweet spot.

In [ ]:
series = get_series(df, store=1, item=1).set_index("ds")

stl = STL(series["y"], period=7, robust=True)  # weekly period for daily data
res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(series.index, series["y"], linewidth=0.6); axes[0].set_title("Observed")
axes[1].plot(series.index, res.trend, linewidth=0.8); axes[1].set_title("Trend")
axes[2].plot(series.index, res.seasonal, linewidth=0.5); axes[2].set_title("Weekly seasonal")
axes[3].plot(series.index, res.resid, linewidth=0.5); axes[3].set_title("Residual")
plt.tight_layout()

## Demand variability across all 500 series

Coefficient of variation (std / mean) per (store, item) tells us which SKUs are "easy" (steady, low CV — Prophet defaults will do fine) vs. "hard" (spiky, high CV — where tuning `changepoint_prior_scale`/`seasonality_prior_scale` in notebook 02 actually pays off, and where the inventory simulation in notebook 03 will want more safety stock).

In [ ]:
stats = (
    df.groupby(["store", "item"])["sales"]
    .agg(mean="mean", std="std")
    .assign(cv=lambda d: d["std"] / d["mean"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(stats["cv"], bins=30, ax=ax)
ax.set_title("Distribution of coefficient of variation across all (store, item) series")
plt.tight_layout()

stats.sort_values("cv", ascending=False).head(10)

## Stationarity check (informative, not a Prophet prerequisite)

Augmented Dickey-Fuller test on a few series — expect to reject stationarity given the visible trend, which is worth stating explicitly in the write-up as the reason Prophet (trend + seasonality decomposition) is a better fit here than a plain ARMA model without differencing.

In [ ]:
sample_keys = list_series_keys(df)[:5]
for store, item in sample_keys:
    s = get_series(df, store, item)["y"]
    stat, pvalue, *_ = adfuller(s)
    print(f"store={store} item={item}: ADF stat={stat:.2f} p-value={pvalue:.4f}")

## Takeaways -> feeding into notebook 02

- Clear multi-year upward trend + weekly and annual seasonality across series -> Prophet is well suited.
- CV varies meaningfully across the 500 series -> tune on a stratified sample (low/medium/high CV) rather than assuming one hyperparameter set fits all, and flag high-CV series as the ones where the inventory simulation matters most.
- Save the variability table for reuse in notebook 03/04 (used to prioritize which SKUs get individual attention).

In [ ]:
from src.data_loader import save_processed

save_processed(stats, "series_variability")